# M2: GAT의 CLV 수준·구성·가격 좌표 임베딩 (Dunnhumby, seed 42)

직전 LightGCN M2와 동일한 `ID(64) + CLV 관계(2) + 가격(1)` layer-0 입력을 단일-head sparse GAT에 이식합니다. `GAT@64`, 동일 총차원의 `GAT@67`, 실제 CLV M2, degree-matched CLV 순열을 고정 100 epoch로 비교합니다. 최종 test와 holdout은 구성하지 않습니다.

네 arm은 순차 실행되며 epoch checkpoint에서 자동 재개됩니다. 판정 기준은 GAT@67 대비 정확도 유지와 가격·구매금액 가중 적중값 개선, 그리고 실제 CLV가 degree-matched shuffle을 이기는지입니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
SOURCE_COMMIT = 'e706d20b28e1996f5008bbd11ea1ab5b61d8c968'
REPO_DIR = '/content/clv-m2-lightgcn-runner'
if len(SOURCE_COMMIT) != 40:
    raise RuntimeError('검토된 소스 커밋을 SOURCE_COMMIT에 고정해야 합니다')
!if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch origin; else git clone {REPO_URL} {REPO_DIR}; fi
!git -C {REPO_DIR} checkout {SOURCE_COMMIT}
%cd {REPO_DIR}
!git rev-parse HEAD

In [ ]:
import json
import torch
from gat_clv_level_composition_price_screen import (
    configure_gat_clv_screen,
    preflight_summary,
    run_gat_clv_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_gat_clv_screen()
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
result_df = run_gat_clv_screen(cfg)

In [ ]:
import pandas as pd
from IPython.display import display

print('1) 절대지표: LightGCN 참고, GAT@64, GAT@67, 실제 CLV, degree-matched shuffle')
display(result_df)
print('2) GAT 내부 대조군별 성과 비교')
display(pd.DataFrame(result_df.attrs['comparison']))
print('3) GAT@67 및 shuffle 대비 실제 CLV Top-10 변경')
display(pd.DataFrame(result_df.attrs['top10_overlap']))
print('4) 사전 판정 규칙 결과')
print(json.dumps(result_df.attrs['screening_reading'], ensure_ascii=False, indent=2))
print('5) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))